# Self-guided lesson — Iterators and Generators in Python

Python often lets us loop through data without thinking about what happens internally:

```python
for value in values:
    ...
```

Behind that simple syntax there are two important ideas:

- **iterables**: objects that can provide values one by one;
- **iterators**: objects that keep track of the current position during iteration.

Generators are a convenient way to create iterators without building a class manually.

In this lesson you will learn:

- the difference between an iterable and an iterator;
- how `iter()` and `next()` work;
- what `StopIteration` means;
- how a `for` loop uses an iterator;
- how generator functions use `yield`;
- why generators are useful for memory-efficient or streaming workflows.

The goal is not to memorize Python internals. The important idea is:

**produce values one at a time instead of building everything at once**

## 1. Iterable vs iterator

An **iterable** is an object that Python can loop over.

Common iterables include:

- lists;
- tuples;
- strings;
- dictionaries;
- sets;

For example, a list is iterable:

In [1]:
numbers = [10, 20, 30, 40]

for number in numbers:
    print(number)

10
20
30
40


An **iterator** is the object that actually produces the next value during the loop.

We can create one explicitly with `iter()`.

In [2]:
numbers = [10, 20, 30, 40]

number_iterator = iter(numbers)

number_iterator

The list and the iterator are related, but they are not the same thing:

```text
list / iterable
      ↓ iter()
iterator
      ↓ next()
next value
```

## 2. `next()` — getting one value at a time

Once we have an iterator, `next()` asks it for the next value.

In [3]:
next(number_iterator)

10

In [4]:
next(number_iterator)

20

In [5]:
next(number_iterator)

30

The iterator remembers where it stopped.

Calling `next()` does **not** restart from the beginning.

In [6]:
next(number_iterator)

40

At this point the iterator has no more values.

If we ask for another one, Python raises `StopIteration`.

In [7]:
try:
    next(number_iterator)
except StopIteration:
    print("The iterator is exhausted.")

The iterator is exhausted.


`StopIteration` is not an error in the usual sense. It is Python's way of saying:

> there are no more values to produce.

Normally, we do not handle this manually because a `for` loop does it for us.

## 3. What a `for` loop is doing conceptually

This:

```python
for value in numbers:
    print(value)
```

is conceptually similar to:

```python
iterator = iter(numbers)

while True:
    try:
        value = next(iterator)
        print(value)
    except StopIteration:
        break
```

You usually write the first version. The second simply reveals the mechanism underneath.

In [8]:
numbers = [1, 2, 3]

iterator = iter(numbers)

while True:
    try:
        value = next(iterator)
        print(value)
    except StopIteration:
        break

1
2
3


Understanding this mechanism will make generators much easier to understand.

## 4. Iterators are consumed

An iterator keeps state, so once its values have been consumed, they are gone.

In [9]:
numbers = [1, 2, 3]

iterator = iter(numbers)

print(list(iterator))
print(list(iterator))

[1, 2, 3]
[]


The first `list(iterator)` consumes all remaining values.

The second one returns an empty list because the iterator is already exhausted.

If we want to iterate again, we create a new iterator:

In [10]:
iterator = iter(numbers)

list(iterator)

[1, 2, 3]

## 5. Generator functions

A **generator function** is a function that uses `yield` instead of returning all values at once.

Consider a countdown:

In [11]:
def countdown(n):
    while n > 0:
        yield n
        n -= 1

Calling the function does not run the complete loop immediately.

It creates a **generator object**:

In [12]:
counter = countdown(5)

counter

<generator object countdown at 0x7fe591063c10>

A generator is also an iterator, so we can use `next()`.

In [13]:
next(counter)

5

In [14]:
next(counter)

4

The function pauses at `yield`.

When `next()` is called again, execution resumes **after the previous `yield`**, using the same local state.

In [15]:
next(counter)

3

A `for` loop is usually the most convenient way to consume the generator:

In [16]:
for number in countdown(5):
    print(number)

5
4
3
2
1


## 6. `yield` vs `return`

A normal function with `return` finishes when it returns a value:

In [17]:
def first_number():
    return 1
    return 2


first_number()

1

A generator can produce several values because each `yield` pauses the function instead of ending it:

In [18]:
def three_numbers():
    yield 1
    yield 2
    yield 3


list(three_numbers())

[1, 2, 3]

A useful mental model is:

```text
return → give back a value and finish
yield  → give back a value and pause
```

## 7. Why generators are useful: lazy evaluation

A list usually creates and stores all its values immediately.

A generator produces values only when they are requested.

Compare:

In [19]:
squares_list = [n ** 2 for n in range(5)]

squares_list

[0, 1, 4, 9, 16]

In [20]:
squares_generator = (n ** 2 for n in range(5))

squares_generator

<generator object <genexpr> at 0x7fe591257450>

The generator expression has not created a list of results. It represents a process that can produce them one at a time.

In [21]:
next(squares_generator)

0

In [22]:
next(squares_generator)

1

This behavior is called **lazy evaluation**.

Instead of:

```text
calculate everything
↓
store everything
↓
use values
```

a generator can work like:

```text
request value
↓
calculate one value
↓
use it
↓
request next value
```

## 8. A small memory comparison

For small datasets, the difference is usually irrelevant.

For very large sequences, storing every value may require much more memory than producing one value at a time.

In [23]:
import sys

numbers_list = [n for n in range(100_000)]
numbers_generator = (n for n in range(100_000))

print("List:", sys.getsizeof(numbers_list), "bytes")
print("Generator:", sys.getsizeof(numbers_generator), "bytes")

List: 800984 bytes
Generator: 192 bytes


`sys.getsizeof()` is only an approximate demonstration here, but it illustrates the idea: the generator object does not store 100,000 integers inside itself.

This does **not** mean generators are always better. If you need to reuse, index, sort, or randomly access all values repeatedly, a list may be more convenient.

## 9. Applied example — processing records one at a time

Imagine we receive lines of data from a large file or external source.

Instead of creating all cleaned records first, a generator can clean and emit one record at a time.

In [24]:
raw_sales = [
    "A001,120.50",
    "A002,95.00",
    "INVALID",
    "A003,210.25"
]

In [25]:
def parse_sales(lines):
    for line in lines:
        parts = line.split(",")

        if len(parts) != 2:
            continue

        product_id, amount = parts

        yield {
            "product_id": product_id,
            "amount": float(amount)
        }

In [26]:
sales_generator = parse_sales(raw_sales)

sales_generator

<generator object parse_sales at 0x7fe59127b210>

In [27]:
for record in sales_generator:
    print(record)

{'product_id': 'A001', 'amount': 120.5}
{'product_id': 'A002', 'amount': 95.0}
{'product_id': 'A003', 'amount': 210.25}


The important point is that `parse_sales()` does not need to build a complete result list before returning anything.

For a four-line example this makes little difference. The same pattern becomes useful when the source contains millions of lines or when values arrive gradually.

## 10. Generators and large files

Python file objects are already iterable.

That means this pattern:

In [28]:
example_lines = [
    "first row",
    "second row",
    "third row"
]

for line in example_lines:
    print(line)

first row
second row
third row


has the same basic idea as reading a real file line by line:

```python
with open("large_file.csv") as file:
    for line in file:
        ...
```

Python does not need to load the entire file into memory just to iterate through its lines.

In data analysis, libraries such as pandas also provide chunked-reading strategies for large datasets. You will usually use those higher-level tools instead of writing a custom iterator class.

## 11. Do we need to build custom iterator classes?

Python allows us to create custom iterator classes by implementing:

```python
__iter__()
__next__()
```

For example, we could recreate something similar to `range()` manually.

That is useful for understanding Python's object model, but for most Data Analytics work it is **not necessary**.

If your goal is simply to produce a sequence of values lazily, a generator function using `yield` is usually much simpler and easier to read.

# 💡 Check for understanding

Create a generator function called:

```python
positive_values(values)
```

that receives an iterable of numbers and yields only values greater than zero.

For example:

```python
values = [10, -4, 0, 8, -2, 5]
```

should produce:

```text
10
8
5
```

### Tasks

1. Write the generator using `yield`.
2. Call the function and inspect the returned object.
3. Use `next()` twice.
4. Create a new generator and consume it with a `for` loop.
5. Convert a fresh generator to a list.

### Optional challenge

Create:

```python
running_total(values)
```

that yields the cumulative sum after each value.

For:

```python
[5, 10, -3, 8]
```

it should yield:

```text
5
15
12
20
```

# Summary

The key ideas are:

1. An **iterable** can provide values for iteration.
2. `iter(iterable)` creates an **iterator**.
3. `next(iterator)` asks for the next value.
4. An exhausted iterator raises `StopIteration`.
5. A `for` loop handles this mechanism automatically.
6. Iterators are **consumed** as values are requested.
7. A generator function uses `yield` to create an iterator easily.
8. `yield` pauses the function and preserves its state.
9. Generators support **lazy evaluation**: values are produced when needed.
10. Generators are especially useful when processing large or streaming sequences one item at a time.

For Data Analytics, the most useful takeaway is not how to implement `__next__()` manually.

It is understanding that Python can process data as a **stream of values** instead of always creating a complete collection first.